In [1]:
%pip install pyarrow pandas

Konfigurasi Berhasil
/home/bumip/orca/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import polars as pl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Tarik Data persis seperti Pipeline
print("[1] Memuat data matriks...")
df_lazy = pl.scan_parquet("/home/bumip/orca/data/silver/*/*/*.parquet", hive_partitioning=True)
df_pl = df_lazy.filter((pl.col("year") == "2024") & (pl.col("month") == "01")).collect()
df_pd = df_pl.to_pandas()

# 2. Log Adapter
df_pd['close_DOGE'] = df_pd['log_DOGE']
df_pd['close_BTC'] = df_pd['log_BTC']
print(f"Data siap: {df_pd.shape[0]} baris.")

# 3. Import Inti Kalman & Komponen Asli
from core.signals.strategies.kalman_mr import KalmanMeanReversion
from core.math import KalmanConfig, AdaptationMode

# --- THE IRON-CLAD MOCK OBJECTS ---
class MockResult:
    def is_err(self): 
        return False  # Bypass validasi dengan memanipulasi Result

class MockSignalConfig:
    name = "Forensic_Kalman"
    version = "1.0"
    
    # Parameter yang ditarik diam-diam oleh kalman_mr.py
    volatility_window = 30    # Untuk hitung std_dev
    hedge_ratio = 1.0         # Untuk _calculate_spread
    entry_z_score = 1.5       # Ambang batas masuk
    exit_z_score = 0.0        # Ambang batas keluar
    stop_loss_z = 5.0         # Ambang batas darurat
    max_position = 1.0
    
    def validate(self):
        return MockResult()

# Kita gunakan KalmanConfig asli agar sesuai dengan kernel Math
math_cfg = KalmanConfig(
    R=1e-3,
    Q=1e-10,   # Kita tes menggunakan Process Noise yang sangat kecil
    initial_value=0.0,
    adaptation_mode=AdaptationMode.NIS_THRESHOLD
)

# 4. Inisialisasi & Eksekusi
strat = KalmanMeanReversion(signal_config=MockSignalConfig(), math_config=math_cfg)

print("\n[2] Mengeksekusi Kalman Filter... (Tunggu sebentar)")
res = strat.generate_signals(df_pd)

if res.is_err():
    print("❌ ERROR KALMAN:", res.unwrap_err())
else:
    df_sig = res.unwrap()
    print("\n✅ --- HASIL FORENSIK KALMAN ---")
    print(f"Max Z-Score Absolut : {df_sig['z_score'].abs().max():.4f}")
    print(f"Min Z-Score Asli    : {df_sig['z_score'].min():.4f}")
    print(f"Max Z-Score Asli    : {df_sig['z_score'].max():.4f}")
    
    print("\nDistribusi Sinyal:")
    if 'signal_type_name' in df_sig.columns:
        print(df_sig['signal_type_name'].value_counts())
    
    # Intip baris di mana Z-Score paling ekstrem
    max_idx = df_sig['z_score'].abs().idxmax()
    cols_to_show = ['timestamp', 'close_DOGE', 'close_BTC', 'z_score', 'signal_type_name']
    
    print("\nBaris dengan Z-Score paling Ekstrem:")
    print(df_sig.loc[max_idx, cols_to_show])

[1] Memuat data matriks...
Data siap: 44640 baris.

[2] Mengeksekusi Kalman Filter... (Tunggu sebentar)

✅ --- HASIL FORENSIK KALMAN ---
Max Z-Score Absolut : 1996.0059
Min Z-Score Asli    : -1996.0059
Max Z-Score Asli    : 927.0267

Distribusi Sinyal:
signal_type_name
NEUTRAL    44622
STOP          15
BUY            2
SELL           1
Name: count, dtype: int64

Baris dengan Z-Score paling Ekstrem:
timestamp           2024-01-01 07:03:00
close_DOGE                    -2.412846
close_BTC                     10.656525
z_score                    -1996.005918
signal_type_name                NEUTRAL
Name: 423, dtype: object


In [26]:
import polars as pl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("[1] Memuat data matriks...")
df_lazy = pl.scan_parquet("/home/bumip/orca/data/silver/*/*/*.parquet", hive_partitioning=True)
df_pl = df_lazy.filter((pl.col("year") == "2024") & (pl.col("month") == "01")).collect()
df_pd = df_pl.to_pandas()

# Log Adapter
df_pd['close_DOGE'] = df_pd['log_DOGE']
df_pd['close_BTC'] = df_pd['log_BTC']
print(f"Data siap: {df_pd.shape[0]} baris.")

from core.signals.strategies.kalman_mr import KalmanMeanReversion
from core.math import KalmanConfig, AdaptationMode

class MockResult:
    def is_err(self): return False

class MockSignalConfig:
    name = "Forensic_Kalman"
    version = "1.0"
    
    # 🛡️ PARAMETER PERSIS SEPERTI SHOTGUN KITA SAAT INI
    volatility_window = 1440  
    hedge_ratio = 1.0         
    entry_z_score = 1.5       
    exit_z_score = 0.0        
    stop_loss_z = 10.0        
    max_position = 1.0
    
    def validate(self): return MockResult()

math_cfg = KalmanConfig(
    R=1e-3,
    Q=1e-10,   
    initial_value=0.0,
    adaptation_mode=AdaptationMode.NIS_THRESHOLD
)

strat = KalmanMeanReversion(signal_config=MockSignalConfig(), math_config=math_cfg)

# MATIKAN TIME BUG SECARA PAKSA UNTUK TES INI
# (Berjaga-jaga jika kode kalman_mr.py Anda di hard drive belum tersimpan)
import time
original_time = time.time
time.time = lambda: 0.0  # Bypass cooldown system

print("\n[2] Mengeksekusi Kalman Filter... (Tunggu sebentar)")
res = strat.generate_signals(df_pd)

# Kembalikan fungsi time ke normal
time.time = original_time

if res.is_err():
    print("❌ ERROR KALMAN:", res.unwrap_err())
else:
    df_sig = res.unwrap()
    print("\n✅ --- HASIL FORENSIK KALMAN V2 ---")
    print(f"Max Z-Score Asli    : {df_sig['z_score'].max():.4f}")
    print(f"Min Z-Score Asli    : {df_sig['z_score'].min():.4f}")
    
    print("\nDistribusi Sinyal:")
    if 'signal_type_name' in df_sig.columns:
        print(df_sig['signal_type_name'].value_counts())

[1] Memuat data matriks...
Data siap: 44640 baris.

[2] Mengeksekusi Kalman Filter... (Tunggu sebentar)

✅ --- HASIL FORENSIK KALMAN V2 ---
Max Z-Score Asli    : 4.1486
Min Z-Score Asli    : -3.4599

Distribusi Sinyal:
signal_type_name
NEUTRAL    44640
Name: count, dtype: int64


In [28]:
import pandas as pd
import polars as pl
from core.signals.strategies.kalman_mr import KalmanMeanReversion
from core.signals.types import SignalConfig
from core.math.kalman import KalmanConfig, AdaptationMode

# ==========================================
# 1. LOAD DATA (HIVE PARTITIONING)
# ==========================================
print("[1] Memuat Data Matriks dari Silver Lake...")
df_lazy = pl.scan_parquet("/home/bumip/orca/data/silver/*/*/*.parquet", hive_partitioning=True)
df_pl = df_lazy.filter((pl.col("year") == "2024") & (pl.col("month") == "01")).collect()

# Convert ke Pandas karena kalman_mr.py minta Pandas DataFrame
df_market = df_pl.to_pandas()
print(f"✅ Data siap: {df_market.shape[0]} baris.")
print(f"📊 Kolom yang tersedia: {df_market.columns.tolist()}")

# ==========================================
# 2. BYPASS NAMA KOLOM (The Bridging)
# ==========================================
# Kalman_mr mencari kolom yang BERAWALAN 'close_'
# Jika di data lu cuma ada 'log_DOGE' atau 'close', kita harus bantu arahkan.
if 'log_DOGE' in df_market.columns and 'close_DOGE' not in df_market.columns:
    df_market['close_DOGE'] = df_market['log_DOGE']
    df_market['close_BTC'] = df_market['log_BTC']
elif 'close' in df_market.columns and 'close_DOGE' not in df_market.columns:
    # Jika hasil join menghasilkan 'close' dan 'close_anchor'
    df_market['close_DOGE'] = df_market['close']
    df_market['close_BTC'] = df_market['close_anchor'] if 'close_anchor' in df_market.columns else df_market['close_BTC']

# ==========================================
# 3. KONFIGURASI BOT
# ==========================================
sig_config = SignalConfig(
    name="Test_Kalman",
    entry_z_score=2.0,
    exit_z_score=0.5,
    stop_loss_z=99999.0,
    max_position=1.0,
    hedge_ratio=1.0,
    volatility_window=1440
)

math_config = KalmanConfig(
    R=1e-3, 
    Q=1e-5, 
    initial_value=0.0,
    adaptation_mode=AdaptationMode.NIS_THRESHOLD
)

bot = KalmanMeanReversion(sig_config, math_config)

# ==========================================
# 4. EKSEKUSI & FORENSIK
# ==========================================
print("\n[2] Menjalankan Kalman Filter...")
res = bot.generate_signals(df_market)

if res.is_err():
    print(f"\n🚨 FATAL ERROR DARI STRATEGI: {res.unwrap_err()}")
else:
    df_signals = res.unwrap()
    
    print("\n✅ --- DISTRIBUSI SINYAL ---")
    print(df_signals['signal_type_name'].value_counts())
    
    print("\n🕵️ --- X-RAY: MENGAPA SINYAL NEUTRAL? (Cek 5 Baris Pertama) ---")
    neutral_metadata = df_signals[df_signals['signal_type_name'] == 'NEUTRAL']['signal_metadata'].head(5)
    for i, meta in enumerate(neutral_metadata):
        print(f"Row {i} Metadata: {meta}")
        
    print("\n📈 --- TOP 5 Z-SCORE TERTINGGI ---")
    top_z = df_signals[['timestamp', 'z_score', 'signal_type_name']].sort_values('z_score', ascending=False).head(5)
    print(top_z)

[1] Memuat Data Matriks dari Silver Lake...
✅ Data siap: 44640 baris.
📊 Kolom yang tersedia: ['timestamp', 'close_BTC', 'close_DOGE', 'log_BTC', 'log_DOGE', 'ret_BTC', 'ret_DOGE', 'vol_BTC_1h', 'vol_DOGE_1h', 'corr_DOGE_BTC_1h', 'vol_BTC_4h', 'vol_DOGE_4h', 'corr_DOGE_BTC_4h', 'vol_BTC_24h', 'vol_DOGE_24h', 'corr_DOGE_BTC_24h', 'beta_DOGE_BTC', 'spread_DOGE', 'z_score_DOGE', 'year', 'month']

[2] Menjalankan Kalman Filter...

✅ --- DISTRIBUSI SINYAL ---
signal_type_name
NEUTRAL    44620
EXIT          20
Name: count, dtype: int64

🕵️ --- X-RAY: MENGAPA SINYAL NEUTRAL? (Cek 5 Baris Pertama) ---
Row 0 Metadata: {'error': 'warmup', 'error_detail': 'Warmup: 1/100', 'status': 'warmup'}
Row 1 Metadata: {'error': 'warmup', 'error_detail': 'Warmup: 2/100', 'status': 'warmup'}
Row 2 Metadata: {'error': 'warmup', 'error_detail': 'Warmup: 3/100', 'status': 'warmup'}
Row 3 Metadata: {'error': 'warmup', 'error_detail': 'Warmup: 4/100', 'status': 'warmup'}
Row 4 Metadata: {'error': 'warmup', 'error_d